# Fatigue et résultats en Ligue des Champions

**Question :** l'équipe qui a eu le plus de repos avant un match de C1 gagne-t-elle plus souvent ?

L'historique utilise toutes les compétitions de `games.csv`; les résultats analysés sont ceux des matchs `CL`. Les indicateurs sont calculés au niveau du club et ne mesurent pas les minutes jouées par chaque joueur.

## Définitions

- **Jours de repos :** jours calendaires entre le match de C1 et le match précédent du club, toutes compétitions confondues.
- **Matchs sur 30 jours :** matchs datés entre les 30 jours avant le match de C1 inclus et la veille du match. Le match étudié n'est pas compté.
- Un match du même jour n'est pas considéré comme antérieur, car l'heure et l'ordre des rencontres ne sont pas garantis.
- **Compétitions par saison :** nombre de `competition_id` distincts auxquels le club a participé pendant la saison entière; c'est descriptif et non une donnée connue en début de saison.
- Pour tester l'effet du repos, les matchs nuls et les matchs où les clubs ont eu le même repos sont exclus.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLEU, ORANGE, VERT, GRIS = '#2a78d6', '#eb6834', '#1baf7a', '#b5b3ad'
ROOT = Path.cwd()
if not (ROOT / 'data' / 'games.csv').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
GAMES_PATH = DATA_DIR / 'games.csv'
EVENTS_PATH = DATA_DIR / 'game_events.csv'
OUTPUT_DIR = ROOT / 'output'
if not GAMES_PATH.exists():
    raise FileNotFoundError(f'Fichier introuvable : {GAMES_PATH}. Place games.csv dans data/.')

games = pd.read_csv(GAMES_PATH)
required = {
    'game_id', 'date', 'season', 'competition_id',
    'home_club_id', 'away_club_id', 'home_club_name', 'away_club_name',
    'home_club_goals', 'away_club_goals',
}
missing = required - set(games.columns)
if missing:
    raise ValueError(f'Colonnes manquantes dans games.csv : {sorted(missing)}')
games['date'] = pd.to_datetime(games['date'], errors='coerce').dt.normalize()
games = games.dropna(subset=['date', 'home_club_id', 'away_club_id', 'competition_id']).copy()
games['competition_id'] = games['competition_id'].astype(str)

home = games[['game_id', 'date', 'season', 'competition_id', 'home_club_id', 'away_club_id', 'home_club_name', 'away_club_name', 'home_club_goals', 'away_club_goals']].copy()
home.columns = ['game_id', 'date', 'season', 'competition_id', 'club_id', 'opponent_id', 'club', 'adversaire', 'buts_pour', 'buts_contre']
home['cote'] = 'Domicile'
away = games[['game_id', 'date', 'season', 'competition_id', 'away_club_id', 'home_club_id', 'away_club_name', 'home_club_name', 'away_club_goals', 'home_club_goals']].copy()
away.columns = ['game_id', 'date', 'season', 'competition_id', 'club_id', 'opponent_id', 'club', 'adversaire', 'buts_pour', 'buts_contre']
away['cote'] = 'Extérieur'
club_games = pd.concat([home, away], ignore_index=True)
club_games['resultat'] = np.select(
    [club_games['buts_pour'] > club_games['buts_contre'], club_games['buts_pour'] < club_games['buts_contre']],
    ['Victoire', 'Défaite'], default='Nul'
)
print(f"{len(games):,} matchs toutes compétitions; {games['competition_id'].nunique():,} compétitions")

## Corriger les scores de tirs au but

Quand `game_events.csv` est disponible, les scores des matchs de C1 identifiés avec une séance (`Shootout`) sont remplacés par le compte des buts (`Goals`) avant les tirs au but. Sinon, les scores de `games.csv` sont conservés et le notebook le signale.

In [ ]:
ucl_ids = set(games.loc[games['competition_id'].eq('CL'), 'game_id'])
shootout_ids = set()
if EVENTS_PATH.exists():
    events = pd.read_csv(EVENTS_PATH, usecols=['game_id', 'type', 'club_id'])
    event_type = events['type'].astype(str).str.strip().str.lower()
    shootout_ids = set(events.loc[event_type.eq('shootout'), 'game_id']) & ucl_ids
    goals = events.loc[event_type.eq('goals')].groupby(['game_id', 'club_id']).size()
    correct = club_games['game_id'].isin(shootout_ids)
    club_games.loc[correct, 'buts_pour'] = [
        goals.get((game_id, club_id), 0)
        for game_id, club_id in zip(club_games.loc[correct, 'game_id'], club_games.loc[correct, 'club_id'])
    ]
    club_games.loc[correct, 'buts_contre'] = [
        goals.get((game_id, opponent_id), 0)
        for game_id, opponent_id in zip(club_games.loc[correct, 'game_id'], club_games.loc[correct, 'opponent_id'])
    ]
    club_games.loc[correct, 'resultat'] = np.select(
        [club_games.loc[correct, 'buts_pour'] > club_games.loc[correct, 'buts_contre'], club_games.loc[correct, 'buts_pour'] < club_games.loc[correct, 'buts_contre']],
        ['Victoire', 'Défaite'], default='Nul'
    )
    print(f'{len(shootout_ids)} match(s) de C1 avec tirs au but corrigé(s) depuis les événements.')
else:
    print('game_events.csv absent : scores enregistrés utilisés; vérifier les séances de tirs au but.')

ucl_teams = club_games.loc[club_games['competition_id'].eq('CL')].copy()
print(f"{ucl_teams['game_id'].nunique():,} matchs de C1 retenus.")

## Indicateurs avant chaque match de C1

Les matchs sont triés par club et par date. La recherche s'arrête au début du jour du match de C1, donc le match étudié et les rencontres du même jour sont exclus.

In [ ]:
competitions_saison = (club_games.groupby(['club_id', 'season'])['competition_id']
                       .nunique().rename('nb_competitions_saison'))
historiques = {club: np.sort(group['date'].to_numpy(dtype='datetime64[ns]'))
              for club, group in club_games.groupby('club_id')}
repos, nb_30j = [], []
for row in ucl_teams.itertuples():
    dates = historiques[row.club_id]
    match_date = np.datetime64(row.date, 'ns')
    end = np.searchsorted(dates, match_date, side='left')
    repos.append(np.nan if end == 0 else int((match_date - dates[end - 1]) / np.timedelta64(1, 'D')))
    start = np.searchsorted(dates, match_date - np.timedelta64(30, 'D'), side='left')
    nb_30j.append(int(end - start))
ucl_teams['jours_repos'] = repos
ucl_teams['matchs_30j_avant'] = nb_30j
ucl_teams = ucl_teams.merge(competitions_saison, left_on=['club_id', 'season'], right_index=True, how='left')

columns = ['game_id', 'date', 'season', 'club_id', 'club', 'opponent_id', 'adversaire', 'cote', 'buts_pour', 'buts_contre', 'resultat', 'jours_repos', 'matchs_30j_avant', 'nb_competitions_saison']
fatigue = ucl_teams[columns].sort_values(['date', 'game_id', 'club_id']).reset_index(drop=True)
fatigue.head()

## Livrable CSV

Une ligne par club et par match de C1; chaque match apparaît donc deux fois, une fois par équipe.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = OUTPUT_DIR / 'fatigue_par_match.csv'
fatigue.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
print(f"CSV créé : {CSV_PATH} ({len(fatigue):,} lignes, {fatigue['game_id'].nunique():,} matchs)")
print(f"Jours de repos manquants : {fatigue['jours_repos'].isna().sum():,}")

## Le club le plus reposé gagne-t-il plus souvent ?

On compare directement les deux clubs de chaque match. Les matchs nuls et ceux où les clubs ont le même repos sont écartés de ce test comparatif.

In [ ]:
pairs = fatigue.pivot(index='game_id', columns='cote', values=['jours_repos', 'resultat'])
pairs.columns = [f'{measure}_{side.lower()}' for measure, side in pairs.columns]
pairs = pairs.dropna(subset=['jours_repos_domicile', 'jours_repos_extérieur'])
difference = pairs['jours_repos_domicile'] - pairs['jours_repos_extérieur']
eligible = difference.ne(0) & pairs['resultat_domicile'].ne('Nul')
home_more_rested = difference.gt(0)
more_rested_won = np.where(
    home_more_rested,
    pairs['resultat_domicile'].eq('Victoire'),
    pairs['resultat_extérieur'].eq('Victoire'),
)
n = int(eligible.sum())
if n:
    wins = int(more_rested_won[eligible.to_numpy()].sum())
    print(f'Le club le plus reposé gagne {wins}/{n} matchs décisifs ({wins / n:.1%}).')
else:
    print('Aucun match comparable après exclusion des nuls et des repos identiques.')

## Taux de victoire selon le repos

Pour chaque catégorie, le taux est le nombre de victoires divisé par toutes les observations club-match de la catégorie. Les nuls restent au dénominateur et ne sont pas des victoires.

In [ ]:
bins = [-np.inf, 3, 5, 8, np.inf]
labels = ['Moins de 3 jours', '3 à 4 jours', '5 à 7 jours', 'Plus de 7 jours']
fatigue['categorie_repos'] = pd.cut(fatigue['jours_repos'], bins=bins, labels=labels, right=False)
rates = (
    fatigue.dropna(subset=['categorie_repos'])
    .groupby('categorie_repos', observed=False)
    .agg(matchs=('resultat', 'size'), victoires=('resultat', lambda values: values.eq('Victoire').sum()))
)
rates['taux_victoire_pct'] = rates['victoires'] / rates['matchs'] * 100
display(rates.round(1))

fig, ax = plt.subplots(figsize=(9, 4.5))
values = rates['taux_victoire_pct'].fillna(0)
bars = ax.bar(labels, values, color=[ORANGE, GRIS, BLEU, VERT], width=0.65)
for bar, (_, row) in zip(bars, rates.iterrows()):
    if row['matchs']:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            f"{row['taux_victoire_pct']:.1f} %\n(n={int(row['matchs'])})",
            ha='center', va='bottom', fontsize=9,
        )
ax.set_ylabel('Victoires parmi les observations club-match (%)')
ax.set_xlabel('Jours de repos avant le match de C1')
ax.set_title('Taux de victoire selon le repos du club')
ax.set_ylim(0, max(100, values.max() + 12))
ax.spines[['top', 'right']].set_visible(False)
plt.xticks(rotation=10, ha='right')
plt.tight_layout()
plt.show()

## Limites

Le nombre de matchs ne mesure ni l'intensité, ni les prolongations, ni les compositions, ni les minutes disputées. Une association observée ne prouve pas que le repos cause une victoire. Si l'historique disponible commence après le début de la saison ou de la carrière d'un club, les jours de repos peuvent être surestimés et les matchs récents sous-comptés.